# 📊 Análise de Tendências Temporais da SpaceX (2006-2022)
**Autor:** Gabriel Garcia (Analista de Tendências Temporais)

## 🔍 Foco da Pesquisa
Como a confiabilidade e o ritmo operacional das missões da SpaceX evoluíram ao longo do tempo? Esta análise busca responder a essa pergunta central por meio de análises estatísticas descritivas, testes de tendências temporais e avaliação de confounders (variáveis de confusão) como a transição entre modelos de foguetes, incorporando intervalos de confiança Wilson Score para a taxa de sucesso anual.

## 1. Setup
Configuração do ambiente, imports de bibliotecas de estatística e visualização e carregamento do dataset oficial versionado `processed_dataset_v1.csv`.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Definindo estilo de plotagem premium
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 15,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'figure.titlesize': 16,
    'figure.dpi': 150
})

# Cores do projeto
colors = {
    'primary': '#0f4c81',      # Azul escuro
    'accent': '#d9534f',       # Coral
    'success': '#2ecc71',      # Verde suave
    'falcon1': '#7f8c8d',      # Cinza
    'falcon9': '#2980b9',      # Azul claro
    'falconheavy': '#9b59b6',  # Roxo
    'dark_grey': '#2c3e50',
    'error_bar': '#c0392b'
}

# Função de Intervalo de Confiança Wilson Score
def wilson_confidence_interval(successes, total, confidence=0.95):
    if total == 0:
        return 0.0, 0.0
    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    p_hat = successes / total
    denominator = 1 + (z**2) / total
    center = (p_hat + (z**2) / (2 * total)) / denominator
    spread = z * np.sqrt((p_hat * (1 - p_hat) / total) + (z**2) / (4 * total**2)) / denominator
    lower = max(0.0, center - spread)
    upper = min(1.0, center + spread)
    return lower * 100, upper * 100

# Carga do Dataset (Garantindo Dataset v1.0)
df = pd.read_csv('../../data/processed/processed_dataset_v1.csv')
df['launch_year'] = df['launch_year'].astype(int)
df['success'] = df['success'].astype(bool)
df['is_reused'] = df['is_reused'].astype(bool)

print(f"Dataset carregado com sucesso!")
print(f"Linhas: {df.shape[0]} (esperado: 192)")
print(f"Colunas: {df.shape[1]}")
df.head()

## 2. Exploração Inicial e Agrupamentos por Ano
Vamos verificar as distribuições anuais, a taxa de sucesso global e gerar a tabela consolidada de resultados anual contendo $n$ (`total_launches`), número de sucessos, reusos e as margens do intervalo de confiança.

In [ ]:
# Taxa de sucesso global no dataset v1
overall_success = df['success'].mean() * 100
print(f"Taxa de Sucesso Global: {overall_success:.2f}% (esperado: 97.40%)")

# Agrupamentos por ano
yearly_stats = df.groupby('launch_year').agg(
    total_launches=('success', 'count'),
    successful_launches=('success', 'sum'),
    reused_boosters=('is_reused', 'sum')
).reset_index()

yearly_stats['success_rate'] = (yearly_stats['successful_launches'] / yearly_stats['total_launches']) * 100
yearly_stats['reuse_rate'] = (yearly_stats['reused_boosters'] / yearly_stats['total_launches']) * 100

yearly_stats['ci_lower'] = yearly_stats.apply(lambda r: wilson_confidence_interval(r['successful_launches'], r['total_launches'])[0], axis=1)
yearly_stats['ci_upper'] = yearly_stats.apply(lambda r: wilson_confidence_interval(r['successful_launches'], r['total_launches'])[1], axis=1)

# Exportar a tabela localmente em CSV para cumprir o deliverable de tabela de resultados
yearly_stats.round(4).to_csv('yearly_stats.csv', index=False)
yearly_stats.round(2)

## 3. Main Analysis (Análise Principal)

### 3.1 Success Rate Over Time (All Rockets) (Q1)
Plotando a taxa de sucesso ao longo dos anos com barras de erros representando o Intervalo de Confiança de $95\%$.

In [ ]:
plt.figure(figsize=(11, 7))
plt.plot(yearly_stats['launch_year'], yearly_stats['success_rate'], marker='o', color=colors['success'], linewidth=2.0, label='Taxa de Sucesso (%)', zorder=3)

# Erros baseados nos ICs de Wilson
yerr_lower = yearly_stats['success_rate'] - yearly_stats['ci_lower']
yerr_upper = yearly_stats['ci_upper'] - yearly_stats['success_rate']
yerr = np.array([yerr_lower, yerr_upper])

plt.errorbar(yearly_stats['launch_year'], yearly_stats['success_rate'], yerr=yerr, fmt='none', ecolor=colors['error_bar'], elinewidth=1.8, capsize=4, capthick=1.5, label='Intervalo de Confiança 95% (Wilson)', zorder=2)

# Delinear as eras com sombreados
plt.axvspan(2006, 2009, color='red', alpha=0.06, label='Era Falcon 1 (Alta incerteza amostral)')
plt.axvspan(2010, 2022, color='green', alpha=0.04, label='Era Falcon 9 & Heavy (Maturidade operacional)')

for idx, row in yearly_stats.iterrows():
    plt.text(row['launch_year'], row['success_rate'] - 5 if row['success_rate'] > 90 else row['success_rate'] + 4, 
             f"{row['success_rate']:.1f}%", ha='center', va='center', fontsize=9, fontweight='bold', zorder=4)
    
plt.title('Evolução da Taxa de Sucesso dos Lançamentos com IC 95% (2006-2022)', pad=15, fontweight='bold')
plt.xlabel('Ano de Lançamento', labelpad=10)
plt.ylabel('Taxa de Sucesso (%)', labelpad=10)
plt.xticks(yearly_stats['launch_year'], rotation=45)
plt.xlim(2005, 2023)
plt.ylim(-10, 115)
plt.legend(loc='lower right', frameon=True)
plt.tight_layout()
plt.savefig('../../graphs/gabriel_success_rate_per_year.png', dpi=300)
plt.show()

### 3.2 Success Rate by Rocket Family + Year (Q4)
Seguindo as orientações do guia, vamos segmentar a taxa de sucesso por modelo de foguete de forma agregada e ao longo dos anos para examinar o efeito do tipo de foguete.

In [ ]:
print("=== TAXA DE SUCESSO AGREGADA POR MODELO DE FOGUETE ===")
rocket_success = df.groupby('rocket_name').agg(
    total_launches=('success', 'count'),
    successes=('success', 'sum')
).reset_index()
rocket_success['success_rate'] = (rocket_success['successes'] / rocket_success['total_launches']) * 100
print(rocket_success.round(2))

print("\n=== TAXA DE SUCESSO POR MODELO E POR ANO ===")
rocket_yearly = df.groupby(['rocket_name', 'launch_year']).agg(
    total_launches=('success', 'count'),
    successes=('success', 'sum')
).reset_index()
rocket_yearly['success_rate'] = (rocket_yearly['successes'] / rocket_yearly['total_launches']) * 100
print(rocket_yearly.round(2))

### 3.3 Launch Frequency Over Time (Q2)

In [ ]:
plt.figure(figsize=(10, 6))
bars = plt.bar(yearly_stats['launch_year'], yearly_stats['total_launches'], color=colors['primary'], alpha=0.85, edgecolor='black', linewidth=0.7)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, height + 0.5, f'{int(height)}', ha='center', va='bottom', fontsize=10, fontweight='bold', color=colors['dark_grey'])
    
plt.title('Ritmo de Lançamentos da SpaceX por Ano (2006-2022)', pad=15, fontweight='bold')
plt.xlabel('Ano de Lançamento', labelpad=10)
plt.ylabel('Quantidade de Lançamentos', labelpad=10)
plt.xticks(yearly_stats['launch_year'], rotation=45)
plt.xlim(2005, 2023)
plt.ylim(0, yearly_stats['total_launches'].max() + 5)
plt.tight_layout()
plt.savefig('../../graphs/gabriel_launches_per_year.png', dpi=300)
plt.show()

### 3.4 Falcon 1 vs Falcon 9+ Trajectory (Análise da Transição tecnológica)
Vamos plotar o mix de produtos anual para deixar evidente a transição tecnológica da era Falcon 1 para a era Falcon 9/Heavy.

In [ ]:
rocket_mix = pd.crosstab(df['launch_year'], df['rocket_name'], normalize='index') * 100
for r in ['Falcon 1', 'Falcon 9', 'Falcon Heavy']:
    if r not in rocket_mix.columns:
        rocket_mix[r] = 0.0
rocket_mix = rocket_mix[['Falcon 1', 'Falcon 9', 'Falcon Heavy']]

plt.figure(figsize=(10, 6))
rocket_mix.plot(kind='bar', stacked=True, color=[colors['falcon1'], colors['falcon9'], colors['falconheavy']], alpha=0.9, edgecolor='black', linewidth=0.5, ax=plt.gca())

plt.title('Composição do Mix de Foguetes Utilizados por Ano (2006-2022)', pad=15, fontweight='bold')
plt.xlabel('Ano de Lançamento', labelpad=10)
plt.ylabel('Proporção do Mix de Foguetes (%)', labelpad=10)
plt.xticks(rotation=45)
plt.ylim(0, 110)
plt.legend(title='Modelo do Foguete', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../../graphs/gabriel_rocket_mix_per_year.png', dpi=300)
plt.show()

# Média comparativa direta recomendada no guia
falcon1_success = df[df['rocket_name'] == 'Falcon 1']['success'].mean()
falcon_modern_success = df[df['rocket_name'] != 'Falcon 1']['success'].mean()
print(f"Taxa de Sucesso Falcon 1: {falcon1_success*100:.2f}% (n={df[df['rocket_name']=='Falcon 1'].shape[0]})")
print(f"Taxa de Sucesso Falcon 9+: {falcon_modern_success*100:.2f}% (n={df[df['rocket_name']!='Falcon 1'].shape[0]})")
print(f"Diferença de Confiabilidade: {(falcon_modern_success - falcon1_success)*100:.2f} pp")

### 3.5 Taxa de Reutilização por Ano
Abaixo plotamos a taxa anual de reuso dos boosters ao longo do tempo para demonstrar como o reuso escalou paralelamente à maturidade operacional.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(yearly_stats['launch_year'], yearly_stats['reuse_rate'], marker='s', linewidth=2.5, color=colors['primary'], label='Taxa de Reuso de Boosters (%)')

# Rótulos nos pontos
for idx, row in yearly_stats.iterrows():
    plt.text(row['launch_year'], row['reuse_rate'] + 3, f"{row['reuse_rate']:.1f}%", ha='center', va='bottom', fontsize=9)
    
plt.title('Evolução da Taxa de Reutilização de Foguetes por Ano (2006-2022)', pad=15, fontweight='bold')
plt.xlabel('Ano de Lançamento', labelpad=10)
plt.ylabel('Taxa de Reutilização (%)', labelpad=10)
plt.xticks(yearly_stats['launch_year'], rotation=45)
plt.xlim(2005, 2023)
plt.ylim(-5, 110)
plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.savefig('../../graphs/gabriel_reuse_rate_per_year.png', dpi=300)
plt.show()

## 4. Análise Estatística e Testes de Tendência

### 4.1 Teste de Tendência: Ritmo de Lançamentos (Q2)
Testando a hipótese nula de que a frequência de lançamentos anuais não possui tendência temporal ao longo dos anos.

In [ ]:
tau_freq, p_freq = stats.kendalltau(yearly_stats['launch_year'], yearly_stats['total_launches'])
res_freq = stats.linregress(yearly_stats['launch_year'], yearly_stats['total_launches'])

print("=== TESTE DE TENDÊNCIA: RITMO DE LANÇAMENTO ===")
print(f"Kendall's Tau: {tau_freq:.4f}")
print(f"P-value: {p_freq:.6e} (Significativo se < 0.05)")
print(f"Inclinação da Reta (Regressão Linear): {res_freq.slope:.4f} lançamentos/ano")
print(f"R-quadrado: {res_freq.rvalue**2:.4f}")

### 4.2 Teste de Tendência: Taxa de Sucesso Geral (Q1 - Todos os Foguetes)
Abaixo realizamos o teste de tendência incluindo a era do Falcon 1.

In [ ]:
tau_succ, p_succ = stats.kendalltau(yearly_stats['launch_year'], yearly_stats['success_rate'])
res_succ = stats.linregress(yearly_stats['launch_year'], yearly_stats['success_rate'])

print("=== TESTE DE TENDÊNCIA: TAXA DE SUCESSO (TODOS OS ANOS) ===")
print(f"Kendall's Tau: {tau_succ:.4f}")
print(f"P-value: {p_succ:.6f} (Significativo se < 0.05)")
print(f"Inclinação da Reta (Regressão Linear): {res_succ.slope:.4f} pp/ano")

### 4.3 Teste de Quebra Estrutural: Tendência da Era Moderna Apenas (2010-2022) (Q4 Confounder)
Isolamos os anos onde operam apenas o Falcon 9 e o Falcon Heavy para checar se a taxa de sucesso apresenta melhoria significativa adicional pós-2010.

In [ ]:
df_modern = df[df['rocket_name'] != 'Falcon 1']
yearly_stats_modern = df_modern.groupby('launch_year').agg(
    total_launches=('success', 'count'),
    successful_launches=('success', 'sum')
).reset_index()
yearly_stats_modern['success_rate'] = (yearly_stats_modern['successful_launches'] / yearly_stats_modern['total_launches']) * 100

tau_succ_mod, p_succ_mod = stats.kendalltau(yearly_stats_modern['launch_year'], yearly_stats_modern['success_rate'])
res_succ_mod = stats.linregress(yearly_stats_modern['launch_year'], yearly_stats_modern['success_rate'])

print("=== TESTE DE TENDÊNCIA: TAXA DE SUCESSO (ERA MODERNA: 2010-2022) ===")
print(f"Kendall's Tau: {tau_succ_mod:.4f}")
print(f"P-value: {p_succ_mod:.6f} (Significativo se < 0.05)")
print(f"Inclinação da Reta (Regressão Linear): {res_succ_mod.slope:.4f} pp/ano")

### 4.4 Teste de Correlação: Experiência da SpaceX vs Confiabilidade (Q3)
Calculamos a correlação Ponto-Bisserial entre `flight_number` e a taxa de sucesso.

In [ ]:
corr_val, p_corr = stats.pointbiserialr(df['flight_number'], df['success'].astype(int))

print("=== CORRELAÇÃO PONTO-BISSERIAL: NÚMERO DO VOO VS SUCESSO ===")
print(f"Coeficiente de Correlação: {corr_val:.4f}")
print(f"P-value: {p_corr:.6f} (Significativo se < 0.05)")

## 5. Conclusões e Interpretações das Análises Temporais

### 5.1 Frequência de Lançamentos (Q2)
A análise mostra um aumento massivo na frequência de lançamentos da SpaceX ao longo dos anos. O teste de Kendall's Tau confirma essa tendência de forma extremamente robusta ($p < 0.001$, $\tau = 0.9234$), indicando um crescimento quase perfeitamente monotônico. A regressão linear mostra um acréscimo médio estimado em **2.23 lançamentos por ano** ao longo de todo o período, acelerando de forma muito marcante nos anos recentes (chegando a 36 lançamentos mapeados em 2022).

### 5.2 Evolução da Confiabilidade (Q1, Q3 & Q4)
1. **Resultado Agregado:** Quando analisado o conjunto de dados completo (incluindo Falcon 1), há uma tendência estatística de aumento da taxa de sucesso por ano com Kendall's Tau de **0.5021** ($p \approx 0.0150$). A correlação entre o número de voo acumulado e a taxa de sucesso é positiva e significante com correlação ponto-bisserial de **0.2463** ($p \approx 0.000573$).
2. **O Efeito Confounder (Modelo de Foguete):** A transição de mix de produtos revela que o Falcon 1 (2006-2009) teve apenas 5 lançamentos e 2 falhas ($60\%$ de sucesso, demonstrando grandes intervalos de incerteza em razão do tamanho amostral). Ao isolarmos apenas os anos modernos (2010 a 2022) onde operam apenas Falcon 9 e Falcon Heavy, a taxa de sucesso estabiliza em patamares elevados (média anual superior a $96\%$).
3. **Tendência da Era Moderna:** Ao testarmos a tendência de evolução da taxa de sucesso apenas para o período 2010-2022, o coeficiente de Kendall's Tau diminui para **0.1343** e deixa de ser estatisticamente significante ($p \approx 0.5933$). Isso revela que **a melhoria de confiabilidade ao longo do tempo foi devida principalmente à transição do Falcon 1 para o Falcon 9/Heavy** ocorrendo em 2010, e que a confiabilidade operacional da SpaceX na era moderna permaneceu estável e em um patamar muito alto, sem apresentar melhoria significativa adicional ano a ano após essa maturação.

### 5.3 Alinhamento com a Pergunta Central de Pesquisa
Com a taxa de reuso subindo de $0\%$ (2006-2015) para mais de $80\%$ (2020-2022), a estabilidade e altíssima confiabilidade observadas na era moderna sugerem fortemente que **a reutilização em larga escala de boosters não comprometeu a segurança ou confiabilidade dos lançamentos**, mantendo taxas de sucesso equivalentes ou superiores às observadas na fase inicial de boosters descartáveis modernos.